In [0]:
%sql
use catalog end_to_end_projects;
use schema sports_equipments_sales;

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import *

In [0]:
dbutils.widgets.text("incremental_load", "", "File Name")
file_name = dbutils.widgets.get("incremental_load")
print(f"Selected File: {file_name}")

In [0]:
source_path = f"/Volumes/end_to_end_projects/sports_equipments_sales/incoming_files/source/{file_name}"
print(f"Incremental Path: {source_path}")

In [0]:
df_incremental = spark.read.csv(source_path, header = True, inferSchema = True)
display(df_incremental.limit(5))
print(f"Count of Records in incremental batch: {df_incremental.count()}")
df_incremental.printSchema()

In [0]:
df_watermark = spark.sql("""
                         select watermark_value 
                         from watermark_control
                         where source_name = "sports_equipment"
                         """
)
watermark_value = df_watermark.first()["watermark_value"]
print(f"Watermark Value: {watermark_value}")

In [0]:
df_incremental_filter = df_incremental.filter(col("last_updated") > watermark_value)
print(f"Count of Original Incremental Records: {df_incremental.count()}")
print(f"Count of Filtered Incremental Records: {df_incremental_filter.count()}")

In [0]:
null_sale_id_count = df_incremental_filter.filter(col("sale_id").isNull()).count()
duplicate_sale_id_count = df_incremental_filter.groupBy("sale_id").count().filter(col("count") > 1)
null_customer_id_count = df_incremental_filter.filter(col("customer_id").isNull()).count()
null_product_id_count = df_incremental_filter.filter(col("product_id").isNull()).count()
null_store_id_count = df_incremental_filter.filter(col("store_id").isNull()).count()
null_salesperson_id_count = df_incremental_filter.filter(col("salesperson_id").isNull()).count()
null_sale_date_count = df_incremental_filter.filter(col("sale_date").isNull()).count()
null_last_updated_count = df_incremental_filter.filter(col("last_updated").isNull()).count()
invalid_quantity_count = df_incremental_filter.filter(col("quantity") <= 0).count()
invalid_unit_price_count = df_incremental_filter.filter(col("unit_price") <= 0).count()
invalid_discount_amount_count = df_incremental_filter.filter(col("discount_amount") < 0).count()
invalid_tax_amount_count = df_incremental_filter.filter(col("tax_amount") < 0).count()
invalid_final_amount_count = df_incremental_filter.filter(col("final_amount") <= 0).count()
invalid_date_check_count = df_incremental_filter.filter(col("sale_date") > to_date(col("last_updated"))).count()
invalid_sale_status_count = df_incremental_filter.filter(col("sale_status").isNull() |~col("sale_status").isin("COMPLETED", "RETURNED", "CANCELLED")).count()

print(f"Null Sale ID Count: {null_sale_id_count}")
print(f"Duplicate Sale ID Count: {duplicate_sale_id_count.count()}")
print(f"Null Customer ID Count: {null_customer_id_count}")
print(f"Null Product ID Count: {null_product_id_count}")
print(f"Null Store ID Count: {null_store_id_count}")
print(f"Null Sales Person ID Count: {null_salesperson_id_count}")
print(f"Null Sale Date Count: {null_sale_date_count}")
print(f"Null Last Updated Count: {null_last_updated_count}")
print(f"Invalid Quantity Count: {invalid_quantity_count}")
print(f"Invalid Unit Price Count: {invalid_unit_price_count}")
print(f"Invalid Discount Amount Count: {invalid_discount_amount_count}")
print(f"Invalid Tax Amount Count: {invalid_tax_amount_count}")
print(f"Invalid Final Amount Count: {invalid_final_amount_count}")
print(f"Invalid Date Check Count: {invalid_date_check_count}")
print(f"Invalid Sale Status Count: {invalid_sale_status_count}")


In [0]:
df_incremental_clean = df_incremental_filter.filter(col("sale_id").isNotNull())\
    .filter(col("customer_id").isNotNull())\
        .filter(col("product_id").isNotNull())\
            .filter(col("store_id").isNotNull())\
                .filter(col("salesperson_id").isNotNull())\
                    .filter(col("sale_date").isNotNull())\
                        .filter(col("last_updated").isNotNull())\
                            .filter(col("quantity") > 0)\
                                .filter(col("unit_price") > 0)\
                                    .filter(col("discount_amount") >= 0)\
                                        .filter(col("tax_amount") >= 0)\
                                            .filter(col("final_amount") > 0)\
                                                .filter(col("sale_date") <= to_date(col("last_updated")))\
                                                    .filter((col("sale_status").isNotNull()) & (col("sale_status").isin("COMPLETED", "RETURNED","CANCELLED")))\
                                                        .withColumn("row_num", row_number().over(Window.partitionBy("sale_id").orderBy(col("last_updated").desc())))\
                                                            .filter(col("row_num") == 1).drop("row_num")
display(df_incremental_clean.limit(5))
df_incremental_clean.printSchema()
print(f"Count of Cleaned Incremental Records: {df_incremental_clean.count()}")

In [0]:
df_incremental_merge = df_incremental_clean.withColumn("ingestion_timestamp", current_timestamp())
display(df_incremental_merge.limit(5))
print(f"Count of Records In Incremental Merge: {df_incremental_merge.count()}")
df_incremental_merge.printSchema()

In [0]:
df_incremental_merge.createOrReplaceTempView("sports_equipment_sales_incremental")

In [0]:
%sql
merge with schema evolution into sports_equipment_sales_silver as target
using sports_equipment_sales_incremental as source
on target.sale_id = source.sale_id

when matched then
update set *

when not matched then
insert *;

In [0]:
df_silver_after_merge = spark.table("sports_equipment_sales_silver")
print(f"Count Of Records in Silver Table After Merge: {df_silver_after_merge.count()}")

In [0]:
duplicate_sale_id_count = df_silver_after_merge.groupBy("sale_id").count().filter(col("count")>1)
print(f"Count of Duplicate IDs After Merge: {duplicate_sale_id_count.count()}")

In [0]:
df_silver_after_merge.printSchema()

In [0]:
%sql
select max(last_updated) as lastest_watermark_value
from sports_equipment_sales_incremental;